In [1]:
# ==========================================
# EXPERIMENT 2 - LOAD DATA
# ==========================================

import os
import glob
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve
)

# CHANGE THIS PATH ONLY IF YOUR DATA FOLDER IS DIFFERENT
DATA_PATH = r"data"

files = sorted(
    glob.glob(
        os.path.join(DATA_PATH, "*.parquet")
    )
)

print("Total files:", len(files))

file_dfs = {}

for file_path in files:

    filename = os.path.basename(file_path)

    df_temp = pd.read_parquet(file_path)

    file_dfs[filename] = df_temp

    print(
        filename,
        "->",
        df_temp.shape
    )

print("\nfile_dfs created successfully.")

Total files: 8
Benign-Monday-no-metadata.parquet -> (155820, 78)
Botnet-Friday-no-metadata.parquet -> (155820, 78)
Bruteforce-Tuesday-no-metadata.parquet -> (155820, 78)
DDoS-Friday-no-metadata.parquet -> (155820, 78)
DoS-Wednesday-no-metadata.parquet -> (155820, 78)
Infiltration-Thursday-no-metadata.parquet -> (155820, 78)
Portscan-Friday-no-metadata.parquet -> (155820, 78)
WebAttacks-Thursday-no-metadata.parquet -> (155820, 78)

file_dfs created successfully.


In [2]:
# ==========================================
# CHECK ALL LABELS
# ==========================================

all_labels = pd.concat(
    [
        df_temp["Label"]
        for df_temp in file_dfs.values()
    ],
    ignore_index=True
)

all_labels = (
    all_labels
    .astype(str)
    .str.strip()
)

print(
    all_labels.value_counts()
)

Label
Benign                        1229416
Web Attack � Brute Force        11760
Web Attack � XSS                 5216
Web Attack � Sql Injection        168
Name: count, dtype: int64


In [3]:
# ==========================================
# SELECT HELD-OUT ATTACK
# ==========================================

HELD_OUT_LABEL = "Web Attack � Sql Injection"

print(
    "Held-out label:",
    HELD_OUT_LABEL
)

heldout_count = (
    all_labels == HELD_OUT_LABEL
).sum()

print(
    "Held-out samples:",
    heldout_count
)

assert heldout_count > 0, (
    "Held-out label not found. "
    "Check the exact label printed in Cell 2."
)

Held-out label: Web Attack � Sql Injection
Held-out samples: 168


In [4]:
# ==========================================
# CREATE TRAINING POOL
# AND ZERO-DAY TEST
# ==========================================

train_parts = []
zero_day_parts = []

for filename, df_temp in file_dfs.items():

    labels = (
        df_temp["Label"]
        .astype(str)
        .str.strip()
    )

    # Everything except held-out attack
    train_data = df_temp[
        labels != HELD_OUT_LABEL
    ].copy()

    train_parts.append(train_data)

    # Only held-out attack
    zero_data = df_temp[
        labels == HELD_OUT_LABEL
    ].copy()

    if len(zero_data) > 0:
        zero_day_parts.append(zero_data)


train_pool_df = pd.concat(
    train_parts,
    ignore_index=True
)

zero_day_df = pd.concat(
    zero_day_parts,
    ignore_index=True
)

print("Training pool:", train_pool_df.shape)
print("Zero-Day test:", zero_day_df.shape)

print("\nTraining labels:")
print(train_pool_df["Label"].value_counts())

print("\nZero-Day labels:")
print(zero_day_df["Label"].value_counts())

Training pool: (1246392, 78)
Zero-Day test: (168, 78)

Training labels:
Label
Benign                      1229416
Web Attack � Brute Force      11760
Web Attack � XSS               5216
Name: count, dtype: int64

Zero-Day labels:
Label
Web Attack � Sql Injection    168
Name: count, dtype: int64


In [5]:
# ==========================================
# CHECK TRAINING HAS BOTH CLASSES
# ==========================================

train_labels = (
    train_pool_df["Label"]
    .astype(str)
    .str.strip()
)

print("Training labels:")
print(train_labels.value_counts())

print(
    "\nHeld-out attack inside training:",
    (train_labels == HELD_OUT_LABEL).sum()
)

assert (
    (train_labels == HELD_OUT_LABEL).sum() == 0
), "ERROR: Held-out attack leaked into training."

assert (
    train_labels.str.upper() == "BENIGN"
).sum() < len(train_labels), (
    "ERROR: Training contains only Benign. "
    "There is no known attack class available."
)

print("\nPASS: Training contains known attacks.")

Training labels:
Label
Benign                      1229416
Web Attack � Brute Force      11760
Web Attack � XSS               5216
Name: count, dtype: int64

Held-out attack inside training: 0

PASS: Training contains known attacks.


In [6]:
# ==========================================
# CHECK TRAINING HAS BOTH CLASSES
# ==========================================

train_labels = (
    train_pool_df["Label"]
    .astype(str)
    .str.strip()
)

print("Training labels:")
print(train_labels.value_counts())

print(
    "\nHeld-out attack inside training:",
    (train_labels == HELD_OUT_LABEL).sum()
)

assert (
    (train_labels == HELD_OUT_LABEL).sum() == 0
), "ERROR: Held-out attack leaked into training."

assert (
    train_labels.str.upper() == "BENIGN"
).sum() < len(train_labels), (
    "ERROR: Training contains only Benign. "
    "There is no known attack class available."
)

print("\nPASS: Training contains known attacks.")

Training labels:
Label
Benign                      1229416
Web Attack � Brute Force      11760
Web Attack � XSS               5216
Name: count, dtype: int64

Held-out attack inside training: 0

PASS: Training contains known attacks.


In [7]:
# ==========================================
# CREATE BINARY TARGET
# ==========================================

train_pool_df["Target"] = (
    train_pool_df["Label"]
    .astype(str)
    .str.strip()
    .str.upper()
    .ne("BENIGN")
    .astype(int)
)

zero_day_df["Target"] = 1

print("Training Target:")
print(train_pool_df["Target"].value_counts())

print("\nZero-Day Target:")
print(zero_day_df["Target"].value_counts())

assert train_pool_df["Target"].nunique() == 2
assert zero_day_df["Target"].nunique() == 1

print("\nPASS: Target created correctly.")

Training Target:
Target
0    1229416
1      16976
Name: count, dtype: int64

Zero-Day Target:
Target
1    168
Name: count, dtype: int64

PASS: Target created correctly.


In [8]:
# ==========================================
# CREATE FEATURES AND TARGET
# ==========================================

# Columns that must NOT be used as features
drop_columns = [
    "Label",
    "Target",
    "Bwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "CWE Flag Count",
    "Fwd Avg Bulk Rate",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd URG Flags"
]

# Features from training data
X_pool = train_pool_df.drop(
    columns=[
        col for col in drop_columns
        if col in train_pool_df.columns
    ]
).copy()

# Features from zero-day data
X_zero_day = zero_day_df.drop(
    columns=[
        col for col in drop_columns
        if col in zero_day_df.columns
    ]
).copy()

# Keep only numeric columns
numeric_columns = X_pool.select_dtypes(
    include="number"
).columns.tolist()

X_pool = X_pool[numeric_columns].copy()

# Make zero-day columns EXACTLY same as training
X_zero_day = X_zero_day[
    numeric_columns
].copy()

# Targets
y_pool = train_pool_df["Target"].copy()
y_zero_day = zero_day_df["Target"].copy()

# Results
print("X_pool shape:", X_pool.shape)
print("X_zero_day shape:", X_zero_day.shape)

print("\nFeature count:", len(numeric_columns))

print("\nTraining Target:")
print(y_pool.value_counts())

print("\nZero-Day Target:")
print(y_zero_day.value_counts())

X_pool shape: (1246392, 67)
X_zero_day shape: (168, 67)

Feature count: 67

Training Target:
Target
0    1229416
1      16976
Name: count, dtype: int64

Zero-Day Target:
Target
1    168
Name: count, dtype: int64


In [9]:
# ==========================================
# KEEP NUMERIC FEATURES
# ==========================================

# Remove Label and Target
feature_columns = [
    col for col in X_pool.columns
    if col not in ["Label", "Target"]
]

X_pool = X_pool[feature_columns].copy()

X_zero_day = zero_day_df[
    feature_columns
].copy()

# Keep only numeric columns
numeric_columns = X_pool.select_dtypes(
    include="number"
).columns

X_pool = X_pool[numeric_columns].copy()

X_zero_day = X_zero_day[
    numeric_columns
].copy()

# Targets
y_pool = train_pool_df["Target"].copy()
y_zero_day = zero_day_df["Target"].copy()

print("Feature shape:", X_pool.shape)
print("Zero-Day feature shape:", X_zero_day.shape)
print("Feature count:", X_pool.shape[1])

print("\nTarget distribution:")
print(y_pool.value_counts())

print("\nZero-Day samples:", len(y_zero_day))

Feature shape: (1246392, 67)
Zero-Day feature shape: (168, 67)
Feature count: 67

Target distribution:
Target
0    1229416
1      16976
Name: count, dtype: int64

Zero-Day samples: 168


In [10]:
# ==========================================
# HANDLE INF AND NaN
# ==========================================

X_pool = X_pool.replace(
    [np.inf, -np.inf],
    np.nan
)

X_zero_day = X_zero_day.replace(
    [np.inf, -np.inf],
    np.nan
)

print(
    "Total NaN in training:",
    X_pool.isna().sum().sum()
)

print(
    "Total NaN in zero-day:",
    X_zero_day.isna().sum().sum()
)

Total NaN in training: 0
Total NaN in zero-day: 0


In [11]:
# ==========================================
# TRAIN / VALIDATION SPLIT
# ==========================================

X_train, X_val, y_train, y_val = train_test_split(
    X_pool,
    y_pool,
    test_size=0.20,
    random_state=42,
    stratify=y_pool
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

print("\ny_train:")
print(y_train.value_counts())

print("\ny_val:")
print(y_val.value_counts())

assert y_train.nunique() == 2
assert y_val.nunique() == 2

print("\nPASS: Both classes present in train and validation.")

X_train: (997113, 67)
X_val: (249279, 67)

y_train:
Target
0    983532
1     13581
Name: count, dtype: int64

y_val:
Target
0    245884
1      3395
Name: count, dtype: int64

PASS: Both classes present in train and validation.


In [12]:
# ==========================================
# TRAINING-BASED IMPUTATION
# ==========================================

train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_val = X_val.fillna(train_medians)
X_zero_day = X_zero_day.fillna(train_medians)

print("Train NaN:",
      X_train.isna().sum().sum())

print("Validation NaN:",
      X_val.isna().sum().sum())

print("Zero-Day NaN:",
      X_zero_day.isna().sum().sum())

Train NaN: 0
Validation NaN: 0
Zero-Day NaN: 0


In [13]:
# ==========================================
# RANDOM FOREST
# ==========================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest trained.")
print("Classes:", rf_model.classes_)

assert len(rf_model.classes_) == 2

Random Forest trained.
Classes: [0 1]


In [14]:
# ==========================================
# ISOLATION FOREST
# ==========================================

X_train_benign = X_train[
    y_train == 0
].copy()

print(
    "Benign training samples:",
    len(X_train_benign)
)

iso_scaler = StandardScaler()

X_train_benign_scaled = (
    iso_scaler.fit_transform(
        X_train_benign
    )
)

iso_model = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

iso_model.fit(
    X_train_benign_scaled
)

print("Isolation Forest trained.")

Benign training samples: 983532
Isolation Forest trained.


In [15]:
# ==========================================
# RANDOM FOREST PROBABILITIES
# ==========================================

rf_train_prob = rf_model.predict_proba(
    X_train
)[:, 1]

rf_val_prob = rf_model.predict_proba(
    X_val
)[:, 1]

rf_zero_day_prob = rf_model.predict_proba(
    X_zero_day
)[:, 1]

print("RF probabilities generated.")

print(
    "Train probability shape:",
    rf_train_prob.shape
)

print(
    "Validation probability shape:",
    rf_val_prob.shape
)

print(
    "Zero-Day probability shape:",
    rf_zero_day_prob.shape
)

RF probabilities generated.
Train probability shape: (997113,)
Validation probability shape: (249279,)
Zero-Day probability shape: (168,)


In [16]:
# ==========================================
# ISOLATION FOREST SCORES
# ==========================================

X_val_scaled = iso_scaler.transform(
    X_val
)

X_zero_day_scaled = iso_scaler.transform(
    X_zero_day
)

iso_val_raw = iso_model.decision_function(
    X_val_scaled
)

iso_zero_day_raw = iso_model.decision_function(
    X_zero_day_scaled
)

# Higher = more anomalous
iso_val_score = -iso_val_raw
iso_zero_day_score = -iso_zero_day_raw

print("Isolation scores generated.")

Isolation scores generated.


In [17]:
# ==========================================
# NORMALIZE ISOLATION SCORES
# ==========================================

iso_min = iso_val_score.min()
iso_max = iso_val_score.max()

iso_val_norm = (
    iso_val_score - iso_min
) / (
    iso_max - iso_min + 1e-12
)

iso_zero_day_norm = (
    iso_zero_day_score - iso_min
) / (
    iso_max - iso_min + 1e-12
)

print("Isolation scores normalized.")

Isolation scores normalized.


In [18]:
# ==========================================
# NORMALIZE ISOLATION SCORES
# ==========================================

iso_min = iso_val_score.min()
iso_max = iso_val_score.max()

iso_val_norm = (
    iso_val_score - iso_min
) / (
    iso_max - iso_min + 1e-12
)

iso_zero_day_norm = (
    iso_zero_day_score - iso_min
) / (
    iso_max - iso_min + 1e-12
)

print("Isolation scores normalized.")

Isolation scores normalized.


In [19]:
# ==========================================
# HYBRID RISK SCORE
# ==========================================

RF_WEIGHT = 0.5
ISO_WEIGHT = 0.5

hybrid_val_score = (
    RF_WEIGHT * rf_val_prob
    +
    ISO_WEIGHT * iso_val_norm
)

hybrid_zero_day_score = (
    RF_WEIGHT * rf_zero_day_prob
    +
    ISO_WEIGHT * iso_zero_day_norm
)

print("Hybrid risk score generated.")

Hybrid risk score generated.


In [21]:
# ==========================================
# DEFINE BEST THRESHOLD
# ==========================================

from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(
    y_val,
    hybrid_val_score
)

f1_scores = (
    2 * precision[:-1] * recall[:-1]
    /
    (
        precision[:-1] + recall[:-1] + 1e-12
    )
)

best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]

print("Best Threshold:", best_threshold)
print("Best Validation F1:", f1_scores[best_index])

Best Threshold: 0.515684507056306
Best Validation F1: 0.9999999999995


In [22]:
# ==========================================
# ZERO-DAY PREDICTIONS
# ==========================================

rf_zero_day_pred = (
    rf_zero_day_prob >= best_threshold
).astype(int)

iso_zero_day_pred = (
    iso_zero_day_norm >= best_threshold
).astype(int)

hybrid_zero_day_pred = (
    hybrid_zero_day_score >= best_threshold
).astype(int)

print("Predictions generated.")

print(
    "\nRF predicted attacks:",
    rf_zero_day_pred.sum()
)

print(
    "Isolation Forest predicted attacks:",
    iso_zero_day_pred.sum()
)

print(
    "Hybrid predicted attacks:",
    hybrid_zero_day_pred.sum()
)

Predictions generated.

RF predicted attacks: 32
Isolation Forest predicted attacks: 0
Hybrid predicted attacks: 8


In [24]:
# ==========================================
# EVALUATION FUNCTION
# ==========================================

def evaluate_model(
    name,
    y_true,
    y_pred,
    y_score
):

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    pr_auc = average_precision_score(
        y_true,
        y_score
    )

    fpr = fp / (
        fp + tn + 1e-12
    )

    fnr = fn / (
        fn + tp + 1e-12
    )

    return {
        "Model": name,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "PR-AUC": pr_auc,
        "FPR": fpr,
        "FNR": fnr
    }

print("Evaluation function ready.")

Evaluation function ready.


In [25]:
# ==========================================
# EXPERIMENT 2 RESULTS
# ==========================================

results_exp2 = pd.DataFrame([

    evaluate_model(
        "Random Forest",
        y_zero_day,
        rf_zero_day_pred,
        rf_zero_day_prob
    ),

    evaluate_model(
        "Isolation Forest",
        y_zero_day,
        iso_zero_day_pred,
        iso_zero_day_norm
    ),

    evaluate_model(
        "Hybrid",
        y_zero_day,
        hybrid_zero_day_pred,
        hybrid_zero_day_score
    )

])

results_exp2

,Model,Precision,Recall,F1,PR-AUC,FPR,FNR
0,Random Forest,1.0,0.190476,0.320000,1.0,0.0,0.809524
1,Isolation Forest,0.0,0.000000,0.000000,1.0,0.0,1.000000
2,Hybrid,1.0,0.047619,0.090909,1.0,0.0,0.952381


In [26]:
# ==========================================
# ZERO-DAY RECALL COMPARISON
# ==========================================

zero_day_recall = pd.DataFrame({

    "Model": [
        "Random Forest",
        "Isolation Forest",
        "Hybrid"
    ],

    "Zero-Day Recall": [

        recall_score(
            y_zero_day,
            rf_zero_day_pred,
            zero_division=0
        ),

        recall_score(
            y_zero_day,
            iso_zero_day_pred,
            zero_division=0
        ),

        recall_score(
            y_zero_day,
            hybrid_zero_day_pred,
            zero_division=0
        )
    ]
})

zero_day_recall

,Model,Zero-Day Recall
0,Random Forest,0.190476
1,Isolation Forest,0.000000
2,Hybrid,0.047619


In [27]:
# ==========================================
# CONFUSION MATRICES
# ==========================================

predictions = {
    "Random Forest": rf_zero_day_pred,
    "Isolation Forest": iso_zero_day_pred,
    "Hybrid": hybrid_zero_day_pred
}

for name, pred in predictions.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    cm = confusion_matrix(
        y_zero_day,
        pred,
        labels=[0, 1]
    )

    print(cm)


Random Forest
[[  0   0]
 [136  32]]

Isolation Forest
[[  0   0]
 [168   0]]

Hybrid
[[  0   0]
 [160   8]]


In [28]:
# ==========================================
# CLASSIFICATION REPORTS
# ==========================================

for name, pred in predictions.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print(
        classification_report(
            y_zero_day,
            pred,
            labels=[0, 1],
            target_names=[
                "Benign",
                "Held-out Attack"
            ],
            zero_division=0
        )
    )


Random Forest
                 precision    recall  f1-score   support

         Benign       0.00      0.00      0.00         0
Held-out Attack       1.00      0.19      0.32       168

       accuracy                           0.19       168
      macro avg       0.50      0.10      0.16       168
   weighted avg       1.00      0.19      0.32       168


Isolation Forest
                 precision    recall  f1-score   support

         Benign       0.00      0.00      0.00       0.0
Held-out Attack       0.00      0.00      0.00     168.0

       accuracy                           0.00     168.0
      macro avg       0.00      0.00      0.00     168.0
   weighted avg       0.00      0.00      0.00     168.0


Hybrid
                 precision    recall  f1-score   support

         Benign       0.00      0.00      0.00         0
Held-out Attack       1.00      0.05      0.09       168

       accuracy                           0.05       168
      macro avg       0.50      0.02   

In [30]:
# ==========================================
# SAVE ALL EXPERIMENT 2 RESULTS
# ==========================================

import os
import joblib

EXP2_PATH = "exp_2_result"

os.makedirs(
    EXP2_PATH,
    exist_ok=True
)

# ------------------------------------------
# 1. RESULTS
# ------------------------------------------

results_exp2.to_csv(
    os.path.join(
        EXP2_PATH,
        "experiment_2_results.csv"
    ),
    index=False
)

zero_day_recall.to_csv(
    os.path.join(
        EXP2_PATH,
        "experiment_2_zero_day_recall.csv"
    ),
    index=False
)

# ------------------------------------------
# 2. RANDOM FOREST MODEL
# ------------------------------------------

joblib.dump(
    rf_model,
    os.path.join(
        EXP2_PATH,
        "experiment2_random_forest.pkl"
    )
)

# ------------------------------------------
# 3. ISOLATION FOREST MODEL
# ------------------------------------------

joblib.dump(
    iso_model,
    os.path.join(
        EXP2_PATH,
        "experiment2_isolation_forest.pkl"
    )
)

# ------------------------------------------
# 4. ISOLATION SCALER
# ------------------------------------------

joblib.dump(
    iso_scaler,
    os.path.join(
        EXP2_PATH,
        "experiment2_isolation_scaler.pkl"
    )
)

# ------------------------------------------
# 5. TRAIN MEDIANS
# ------------------------------------------

joblib.dump(
    train_medians,
    os.path.join(
        EXP2_PATH,
        "experiment2_train_medians.pkl"
    )
)

# ------------------------------------------
# 6. HYBRID CONFIGURATION
# ------------------------------------------

hybrid_config = {
    "rf_weight": RF_WEIGHT,
    "isolation_weight": ISO_WEIGHT,
    "best_threshold": float(best_threshold),
    "held_out_label": HELD_OUT_LABEL,
    "feature_columns": numeric_columns
}

joblib.dump(
    hybrid_config,
    os.path.join(
        EXP2_PATH,
        "experiment2_hybrid_config.pkl"
    )
)

# ------------------------------------------
# 7. PRINT SAVED FILES
# ------------------------------------------

print("=" * 60)
print("EXPERIMENT 2 SAVED SUCCESSFULLY")
print("=" * 60)

for filename in os.listdir(EXP2_PATH):
    print(filename)

print("\nFolder:", EXP2_PATH)

EXPERIMENT 2 SAVED SUCCESSFULLY
experiment2_hybrid_config.pkl
experiment2_isolation_forest.pkl
experiment2_isolation_scaler.pkl
experiment2_random_forest.pkl
experiment2_train_medians.pkl
experiment_2_results.csv
experiment_2_zero_day_recall.csv

Folder: exp_2_result


In [31]:
# ==========================================
# EXPERIMENT 2 - MODEL HEALTH CHECK
# ==========================================

print("=" * 60)
print("EXPERIMENT 2 MODEL CHECK")
print("=" * 60)

print("\n1. Random Forest classes:")
print(rf_model.classes_)

print("\n2. Training class distribution:")
print(y_train.value_counts())

print("\n3. Zero-Day samples:")
print(len(y_zero_day))

print("\n4. Zero-Day actual labels:")
print(y_zero_day.value_counts())

print("\n5. RF predictions:")
print(pd.Series(rf_zero_day_pred).value_counts())

print("\n6. Isolation Forest predictions:")
print(pd.Series(iso_zero_day_pred).value_counts())

print("\n7. Hybrid predictions:")
print(pd.Series(hybrid_zero_day_pred).value_counts())

print("\n8. Hybrid threshold:")
print(best_threshold)

print("\n9. Final Results:")
display(results_exp2)

print("\n" + "=" * 60)
print("CHECK COMPLETE")
print("=" * 60)

EXPERIMENT 2 MODEL CHECK

1. Random Forest classes:
[0 1]

2. Training class distribution:
Target
0    983532
1     13581
Name: count, dtype: int64

3. Zero-Day samples:
168

4. Zero-Day actual labels:
Target
1    168
Name: count, dtype: int64

5. RF predictions:
0    136
1     32
Name: count, dtype: int64

6. Isolation Forest predictions:
0    168
Name: count, dtype: int64

7. Hybrid predictions:
0    160
1      8
Name: count, dtype: int64

8. Hybrid threshold:
0.515684507056306

9. Final Results:


,Model,Precision,Recall,F1,PR-AUC,FPR,FNR
0,Random Forest,1.0,0.190476,0.320000,1.0,0.0,0.809524
1,Isolation Forest,0.0,0.000000,0.000000,1.0,0.0,1.000000
2,Hybrid,1.0,0.047619,0.090909,1.0,0.0,0.952381



CHECK COMPLETE
